In [1]:
import argparse, json, pathlib, sys
from typing import Any, List, Sequence, Union

Json = Union[dict, list, str, int, float, bool, None]
JsonPath = Sequence[Union[str, int]]

In [2]:
def search_key_json(
    node: Json,
    target: str,
    contains: bool,
    path_so_far: Sequence[Union[str, int]],
    results: List[tuple[JsonPath, str, Json]],
    ignore_keys: set[str] = set(),
) -> None:
    """Search for matching keys in a JSON object, with support for ignoring keys."""
    if isinstance(node, dict):
        for key, value in node.items():
            if key in ignore_keys:
                continue

            match = (
                target.lower() in key.lower()
                if contains and isinstance(key, str)
                else key == target
            )
            if match:
                results.append((path_so_far + [key], key, value))

            search_key_json(value, target, contains, path_so_far + [key], results, ignore_keys)

    elif isinstance(node, list):
        for idx, item in enumerate(node):
            search_key_json(item, target, contains, path_so_far + [idx], results, ignore_keys)



In [3]:
def to_newtonsoft(
    path: JsonPath,
    *,
    escape_backslashes: bool = False,
    safe: bool = False,
) -> str:
    """
    Convert a `search_json` path to a Newtonsoft‑style accessor.

    Parameters
    ----------
    escape_backslashes : bool
        If True, every back‑slash is doubled *after* the key has been
        quote‑escaped, so the whole string can be pasted into a plain
        C# string literal.
    safe : bool
        If True, inserts the null‑conditional operator `?` in front of
        every '[' segment.
    """
    parts = ["root"]
    for seg in path:
        prefix = "?[" if safe else "["
        if isinstance(seg, int):
            parts.append(f"{prefix}{seg}]")
            continue

        # 1️⃣ start with the raw key text
        txt = seg
        # 2️⃣ escape double‑quotes first (this introduces back‑slashes)
        txt = txt.replace('"', r'\"')
        # 3️⃣ *then* optionally double every back‑slash
        if escape_backslashes:
            txt = txt.replace("\\", r"\\")
        parts.append(f'{prefix}"{txt}"]')
    return "".join(parts)


import re

import json
def print_newtonsoft(json_data, newtonsoft_path: str) -> None:
    """
    Evaluate a Newtonsoft-style path (with or without '?' safety operators)
    and pretty-print the value it resolves to.
    """

    # Strip optional '?' operators that come from `to_newtonsoft(..., safe=True)`
    newtonsoft_path = newtonsoft_path.replace("?", "")

    # Match either ["string key"] or [123] segments
    parts = re.findall(
        r'\[\s*"((?:[^"\\]|\\.)*)"\s*\]'   # group 1: string key
        r'|\[\s*(\d+)\s*\]',              # group 2: list index
        newtonsoft_path
    )

    current = json_data
    for key, idx in parts:
        if key:                           # dict access
            key = bytes(key, "utf-8").decode("unicode_escape")
            if not isinstance(current, dict) or key not in current:
                print(f"❌ Key '{key}' not found at this level.")
                return
            current = current[key]
        else:                             # list index access
            i = int(idx)
            if not isinstance(current, list) or i >= len(current):
                print(f"❌ Index [{i}] out of range.")
                return
            current = current[i]

    print("\n✅ Final value:")
    print(json.dumps(current, indent=2, ensure_ascii=False))


In [4]:
import json

# Load JSON data
with open("my.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [6]:
target_key = "host"
ignore_keys = {"linkName", "name", "__typename", "categoryName", "markupType", "viewType"} 

matches = []
search_key_json(
    data,
    target=target_key,
    contains=True,
    path_so_far=[],
    results=matches,
    ignore_keys=ignore_keys,
)

for path, key, value in matches:
    path_str = to_newtonsoft(path, safe=True)
    preview = json.dumps(value, ensure_ascii=False)
    preview = (preview[:47] + "...") if len(preview) > 50 else preview
    print(f"🔑 Key Match: {key}\n📍 Path: {path_str}\n🔍 Value Preview: {preview}\n")



🔑 Key Match: aboutThisHost
📍 Path: root?["PropertyInfo:33941257"]?["propertyContentSectionGroups({\"searchCriteria\":{\"primary\":{\"dateRange\":null,\"destination\":{\"regionId\":\"553248635976434037\"},\"rooms\":[{\"adults\":2}]}}})"]?["aboutThisHost"]
🔍 Value Preview: {"__typename": "PropertyContentSectionGroup", "...

